# Hypatia: Hardware-Aware Symbolic Compiler for PyTorch

[![GitHub](https://img.shields.io/badge/GitHub-mahreman%2Fhypatia-blue)](https://github.com/mahreman/hypatia)
[![License](https://img.shields.io/badge/License-MIT-green)](https://github.com/mahreman/hypatia/blob/main/LICENSE)

**Hypatia** uses E-Graph equality saturation (via Rust's `egg` crate) to discover optimal operator fusion patterns in neural network computation graphs. It integrates as a drop-in `torch.compile` backend.

## What This Notebook Demonstrates

1. **Installation** from source (Rust + Python)
2. **E-Graph Optimization** — symbolic rewriting of computation graphs
3. **Hardware Detection & FLOPs Profiler** — GPU capability analysis
4. **Auto-Tuner** — automatic strategy selection
5. **Fused Modules** — GPU-accelerated fused kernels
6. **Benchmark: Hypatia vs TorchInductor** — fair GPU comparison
7. **INT4 Quantization** — compression with cosine similarity validation
8. **Interactive Dashboard** — HTML benchmark report

---

> **GPU Required**: This notebook needs a GPU runtime.
> Go to `Runtime > Change runtime type > T4 GPU`

## 0. Setup & Installation

Install Rust toolchain and build Hypatia from source. This takes ~3-5 minutes on first run.

In [ ]:
%%bash
# Step 1: Install Rust toolchain
if ! command -v rustc &> /dev/null; then
    echo "Installing Rust..."
    curl --proto '=https' --tlsv1.2 -sSf https://sh.rustup.rs | sh -s -- -y 2>&1 | tail -3
else
    echo "Rust already installed"
fi
source $HOME/.cargo/env
echo "Rust: $(rustc --version)"

In [ ]:
%%bash
source $HOME/.cargo/env

# Step 2: Clone Hypatia (use p3-p6-enhancements branch with all fixes)
if [ ! -d "hypatia" ]; then
    echo "Cloning Hypatia repository..."
    git clone --depth 1 -b p3-p6-enhancements https://github.com/mahreman/hypatia.git 2>&1 | tail -3
else
    echo "Hypatia already cloned — pulling latest changes..."
    cd hypatia && git pull origin p3-p6-enhancements 2>&1 | tail -5 && cd ..
fi

# Step 3: Install build tools
pip install -q "maturin[patchelf]" ninja patchelf

# Step 4: Build Rust extension with --skip-auditwheel to avoid libopenblas repair issues
cd hypatia/hypatia_core
PYTHON_BIN=$(which python3)
echo "Python interpreter: $PYTHON_BIN"
echo "Python version: $(python3 --version)"
echo "Building Hypatia Rust core (this takes ~2-3 minutes)..."

BUILD_OUTPUT=$(maturin build --release --interpreter "$PYTHON_BIN" --skip-auditwheel 2>&1)
BUILD_EXIT=$?
echo "$BUILD_OUTPUT" | tail -10
echo ""

# Check for actual wheel file as the real success indicator
WHEEL_FILE=$(ls target/wheels/*.whl 2>/dev/null | head -1)

if [ -n "$WHEEL_FILE" ]; then
    echo "Found wheel: $WHEEL_FILE"
    echo "Installing..."
    pip install "$WHEEL_FILE" --force-reinstall --no-deps 2>&1 | tail -3
    # Verify import works
    if python3 -c "from _hypatia_core import optimize_ast; print('Rust core: OK')" 2>&1; then
        echo "=== Build SUCCESS ==="
    else
        echo "=== Wheel installed but import failed ==="
        echo "Python-only features will still work."
    fi
else
    echo "=== Build FAILED — no wheel produced ==="
    echo "maturin exit code: $BUILD_EXIT"
    echo ""
    echo "Falling back to Python-only mode."
    echo "Features: profiler, auto-tuner, dashboard, fused modules still work."
fi

In [ ]:
import sys
import os

# Add hypatia to path
hypatia_path = os.path.join(os.getcwd(), 'hypatia', 'hypatia_core')
if hypatia_path not in sys.path:
    sys.path.insert(0, hypatia_path)

import torch
import torch.nn as nn
import hypatia_core

print(f"PyTorch:  {torch.__version__}")
print(f"CUDA:     {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU:      {torch.cuda.get_device_name(0)}")
    props = torch.cuda.get_device_properties(0)
    print(f"VRAM:     {props.total_memory / 1e9:.1f} GB")
    print(f"SM:       {props.major}.{props.minor}")
print(f"Hypatia:  v{hypatia_core.__version__}")
print(f"Rust core: {'AVAILABLE' if hypatia_core._RUST_AVAILABLE else 'NOT available (Python-only features will work)'}")

---
## 1. E-Graph Optimization: Symbolic Rewriting

Hypatia's E-graph engine (Rust) takes S-expressions representing computation graphs and discovers optimal fusions via equality saturation.

> **Requires Rust core.** If the build above failed, this section will be skipped.

In [ ]:
if not hypatia_core._RUST_AVAILABLE:
    print("Skipping: Rust core not available. Re-run the build cell above.")
else:
    from hypatia_core import optimize_ast, parse_expr

    examples = [
        ("Linear + ReLU fusion",
         "(relu (linear w b x))",
         "relu(linear(...)) -> fused_linear_relu(...)"),
        ("Multi-layer MLP fusion",
         "(relu (linear w2 b2 (relu (linear w1 b1 x))))",
         "Both layers fused independently"),
        ("Attention -> SDPA",
         "(attention q k v)",
         "Multi-head attention -> Scaled Dot Product Attention"),
        ("GELU MLP fusion (Transformer advantage)",
         "(gelu (linear w b x))",
         "GELU+Linear -> fused_gelu_mlp"),
        ("Algebraic simplification",
         "(add (mul x 1) 0)",
         "Identity elimination: x*1 + 0 -> x"),
    ]

    for title, expr, explanation in examples:
        result = optimize_ast(expr)
        print(f"--- {title} ---")
        print(f"  Input:  {expr}")
        print(f"  Output: {result}")
        print(f"  {explanation}\n")

### Visualization: Expression Tree

In [ ]:
if hypatia_core._RUST_AVAILABLE:
    from hypatia_core import print_expr_tree

    expr = "(relu (linear w2 b2 (relu (linear w1 b1 x))))"
    print("=== Before Optimization ===")
    print_expr_tree(expr)

    optimized = optimize_ast(expr)
    print("\n=== After Optimization ===")
    print_expr_tree(optimized)
else:
    print("Skipping: Rust core not available.")

---
## 2. Hardware Detection & FLOPs Profiler

Automatic GPU capability analysis with roofline model classification.

> **Works without Rust core** — these are pure Python features.

In [ ]:
from hypatia_core import detect_hardware, profile_model, roofline_analysis

# Detect GPU capabilities
hw = detect_hardware()
print("=== Hardware Detection ===")
print(f"GPU: {hw.gpu_name}")
print(f"Compute Capability: SM {hw.gpu_compute_capability[0]}.{hw.gpu_compute_capability[1]}")
print(f"Tensor Cores: {hw.tensor_core_gen}")
print(f"VRAM: {hw.gpu_memory_gb:.1f} GB")
print(f"Peak FP32: {hw.peak_tflops_fp32:.1f} TFLOPS")
print(f"Peak FP16: {hw.peak_tflops_fp16:.1f} TFLOPS")
print(f"Memory BW: {hw.memory_bandwidth_gbs:.0f} GB/s")
print(f"FP16 support: {hw.supports_fp16}")
print(f"BF16 support: {hw.supports_bf16}")

In [ ]:
# Profile a model
class DemoMLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(768, 3072),
            nn.GELU(),
            nn.Linear(3072, 768),
        )
    def forward(self, x):
        return self.net(x)

model = DemoMLP()
profile = profile_model(model, input_shape=(1, 128, 768))

print(f"=== Model Profile ===")
print(f"Total FLOPs: {profile.total_flops / 1e9:.2f} GFLOPs")
print(f"Total Params: {profile.total_params:,}")
print(f"Memory (FP32): {profile.total_params * 4 / 1e6:.1f} MB")
print(f"\nPer-layer breakdown:")
for layer in profile.layers:
    print(f"  {layer.name}: {layer.flops/1e6:.1f} MFLOPs ({layer.module_type})")

# Roofline analysis (uses hardware info from profile)
roof = roofline_analysis(profile)
print(f"\n=== Roofline Analysis ===")
print(f"Ridge Point: {roof['ridge_point']:.1f} FLOPs/byte")
print(f"Arithmetic Intensity: {roof['arithmetic_intensity']:.1f} FLOPs/byte")
print(f"Bottleneck: {roof['bottleneck']}-bound")
print(f"\n{roof['recommendation']}")

---
## 3. Auto-Tuner: Automatic Strategy Selection

The auto-tuner analyzes model architecture and hardware to select the optimal optimization strategy.

In [ ]:
from hypatia_core import quick_tune

# Quick tune for MLP
model = DemoMLP()
config = quick_tune(model, input_shape=(1, 128, 768))

print("=== Quick Tune: MLP ===")
print(f"Strategy: {config.strategy_name}")
print(f"Mode: {config.mode}")
print(f"Mixed Precision: {config.mixed_precision}")
print(f"Enable Fusion: {config.enable_fusion}")
print(f"Chain Compile: {config.chain_compile}")
print(f"Quantize: {config.quantize}")
print(f"\nDecision log:")
for entry in config.search_log:
    print(f"  - {entry}")

In [ ]:
# Quick tune for Transformer
class MiniTransformer(nn.Module):
    def __init__(self, d_model=512, nhead=8, num_layers=4):
        super().__init__()
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=nhead, dim_feedforward=2048, batch_first=True
        )
        self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        self.fc = nn.Linear(d_model, 10)

    def forward(self, x):
        return self.fc(self.encoder(x).mean(dim=1))

transformer = MiniTransformer()
config_t = quick_tune(transformer, input_shape=(4, 64, 512))

print("=== Quick Tune: Transformer ===")
print(f"Strategy: {config_t.strategy_name}")
print(f"Precision: {config_t.mixed_precision}")
print(f"Fusion: {config_t.enable_fusion}")
print(f"\nDecision log:")
for entry in config_t.search_log:
    print(f"  - {entry}")

---
## 4. Fused Modules: GPU-Accelerated Kernels

Hypatia provides fused modules that combine multiple operations into single GPU kernels.

In [ ]:
import time
from hypatia_core import FusedGeluMLP

device = 'cuda' if torch.cuda.is_available() else 'cpu'

d_model, d_ff = 512, 2048
batch, seq = 4, 64

# Standard PyTorch
standard_mlp = nn.Sequential(
    nn.Linear(d_model, d_ff),
    nn.GELU(),
    nn.Linear(d_ff, d_model),
).to(device).eval()

# Hypatia Fused (in_features, hidden_features, out_features)
fused_mlp = FusedGeluMLP(d_model, d_ff, d_model).to(device).eval()

x = torch.randn(batch, seq, d_model, device=device)

def bench(model, inp, name, n_warmup=10, n_iter=100):
    with torch.no_grad():
        for _ in range(n_warmup):
            model(inp)
        if device == 'cuda':
            torch.cuda.synchronize()
        t0 = time.perf_counter()
        for _ in range(n_iter):
            model(inp)
        if device == 'cuda':
            torch.cuda.synchronize()
        elapsed = (time.perf_counter() - t0) / n_iter * 1000
    print(f"  {name}: {elapsed:.3f} ms")
    return elapsed

print("=== GELU MLP Benchmark (GPU) ===")
t_std = bench(standard_mlp, x, "Standard PyTorch")
t_fused = bench(fused_mlp, x, "Hypatia Fused")
print(f"  Speedup: {t_std/t_fused:.2f}x")

---
## 5. Hypatia vs TorchInductor: Fair GPU Benchmark

The critical comparison: `torch.compile(backend='hypatia')` vs `torch.compile(backend='inductor')` on the **same GPU**.

> **Requires Rust core** for the Hypatia backend. Inductor comparison runs regardless.

In [ ]:
import gc
import statistics

device = 'cuda'
WARMUP = 5
ITERS = 50

class SmallMLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(784, 256), nn.ReLU(),
            nn.Linear(256, 128), nn.ReLU(),
            nn.Linear(128, 10),
        )
    def forward(self, x):
        return self.net(x)

class TransformerBlock(nn.Module):
    def __init__(self, d_model=512, nhead=8, dim_ff=2048):
        super().__init__()
        self.attn = nn.MultiheadAttention(d_model, nhead, batch_first=True)
        self.norm1 = nn.LayerNorm(d_model)
        self.ff = nn.Sequential(
            nn.Linear(d_model, dim_ff), nn.GELU(),
            nn.Linear(dim_ff, d_model),
        )
        self.norm2 = nn.LayerNorm(d_model)
    def forward(self, x):
        attn_out, _ = self.attn(x, x, x)
        x = self.norm1(x + attn_out)
        x = self.norm2(x + self.ff(x))
        return x

def gpu_bench(model, inp, label):
    try:
        model.eval()
        with torch.no_grad():
            for _ in range(WARMUP):
                model(inp)
            torch.cuda.synchronize()
            times = []
            for _ in range(ITERS):
                torch.cuda.synchronize()
                t0 = time.perf_counter()
                model(inp)
                torch.cuda.synchronize()
                times.append((time.perf_counter() - t0) * 1000)
        mean_ms = statistics.mean(times)
        print(f"    {label}: {mean_ms:.3f} ms")
        return mean_ms
    except Exception as e:
        print(f"    {label}: FAILED - {e}")
        return None

models_to_bench = [
    ("Small MLP", SmallMLP, (32, 784)),
    ("Transformer Block", TransformerBlock, (32, 64, 512)),
]

results = {}
for name, model_cls, shape in models_to_bench:
    print(f"\n--- {name} ---")
    inp = torch.randn(shape, device=device)

    # Vanilla GPU
    torch._dynamo.reset()
    m = model_cls().to(device).eval()
    vanilla = gpu_bench(m, inp, "Vanilla GPU")
    del m; torch.cuda.empty_cache(); gc.collect()

    # TorchInductor
    torch._dynamo.reset()
    m = model_cls().to(device).eval()
    print("    Compiling with Inductor...")
    m_ind = torch.compile(m, backend="inductor", mode="max-autotune")
    inductor = gpu_bench(m_ind, inp, "Inductor (max-autotune)")
    del m, m_ind; torch.cuda.empty_cache(); gc.collect()

    # Hypatia (only if Rust core available)
    hypatia_ms = None
    if hypatia_core._RUST_AVAILABLE:
        torch._dynamo.reset()
        m = model_cls().to(device).eval()
        print("    Compiling with Hypatia...")
        m_hyp = torch.compile(m, backend="hypatia")
        hypatia_ms = gpu_bench(m_hyp, inp, "Hypatia")
        del m, m_hyp; torch.cuda.empty_cache(); gc.collect()
    else:
        print("    Hypatia: SKIPPED (Rust core not available)")

    results[name] = {"vanilla": vanilla, "inductor": inductor, "hypatia": hypatia_ms}

# Summary
print("\n" + "="*65)
print("  SUMMARY: Hypatia vs Inductor")
print("="*65)
print(f"  {'Model':<25} {'Vanilla':>10} {'Inductor':>10} {'Hypatia':>10} {'Hyp/Ind':>10}")
print("  " + "-"*60)
for name, r in results.items():
    van = f"{r['vanilla']:.3f}ms" if r['vanilla'] else "N/A"
    ind = f"{r['inductor']:.3f}ms" if r['inductor'] else "N/A"
    hyp = f"{r['hypatia']:.3f}ms" if r['hypatia'] else "N/A"
    ratio = f"{r['hypatia']/r['inductor']:.2f}x" if r['inductor'] and r['hypatia'] else "N/A"
    print(f"  {name:<25} {van:>10} {ind:>10} {hyp:>10} {ratio:>10}")
print("\n  Hyp/Ind < 1.0 = Hypatia faster | > 1.0 = Inductor faster")

---
## 5b. Scaling Benchmark: Larger Models

How does Hypatia's E-graph advantage scale with model complexity?
We test progressively larger architectures on the same T4 GPU.

| Model | Params | Why interesting |
|-------|--------|-----------------|
| GPT-2 Small (4-layer) | ~30M | Realistic decoder-only Transformer |
| Deep MLP (6-layer) | ~25M | Tests where Inductor excels |
| Wide Transformer (d=1024) | ~50M | Larger attention + FFN blocks |
| Multi-Head Encoder (12-layer) | ~85M | BERT-base scale |

> **Note**: Compilation time is one-time cost. We measure **inference latency only**.

In [ ]:
import gc, time, statistics
import torch
import torch.nn as nn

device = 'cuda'
WARMUP = 5
ITERS = 50

# ============================================================
# Model Definitions
# ============================================================

class DeepMLP(nn.Module):
    """6-layer MLP with GELU — tests pure feed-forward fusion."""
    def __init__(self, d=1024):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(d, d*4), nn.GELU(),
            nn.Linear(d*4, d*2), nn.GELU(),
            nn.Linear(d*2, d*4), nn.GELU(),
            nn.Linear(d*4, d*2), nn.GELU(),
            nn.Linear(d*2, d), nn.GELU(),
            nn.Linear(d, 10),
        )
    def forward(self, x): return self.net(x)

class GPT2Block(nn.Module):
    """Single GPT-2 style decoder block (causal self-attention + MLP)."""
    def __init__(self, d_model=768, nhead=12, dim_ff=3072):
        super().__init__()
        self.ln1 = nn.LayerNorm(d_model)
        self.attn = nn.MultiheadAttention(d_model, nhead, batch_first=True)
        self.ln2 = nn.LayerNorm(d_model)
        self.mlp = nn.Sequential(
            nn.Linear(d_model, dim_ff), nn.GELU(),
            nn.Linear(dim_ff, d_model),
        )
    def forward(self, x):
        # Pre-norm architecture (GPT-2 style)
        h = self.ln1(x)
        attn_out, _ = self.attn(h, h, h)
        x = x + attn_out
        x = x + self.mlp(self.ln2(x))
        return x

class GPT2Small(nn.Module):
    """4-layer GPT-2 style model (~30M params)."""
    def __init__(self, d_model=768, nhead=12, n_layers=4):
        super().__init__()
        self.blocks = nn.Sequential(*[
            GPT2Block(d_model, nhead, d_model*4) for _ in range(n_layers)
        ])
        self.ln_f = nn.LayerNorm(d_model)
        self.head = nn.Linear(d_model, 10)
    def forward(self, x):
        x = self.blocks(x)
        return self.head(self.ln_f(x).mean(dim=1))

class WideTransformer(nn.Module):
    """Wide Transformer encoder (d=1024, 6 layers, ~50M params)."""
    def __init__(self, d_model=1024, nhead=16, n_layers=6):
        super().__init__()
        layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=nhead, dim_feedforward=d_model*4,
            batch_first=True, norm_first=True, activation='gelu',
        )
        self.encoder = nn.TransformerEncoder(layer, num_layers=n_layers)
        self.head = nn.Linear(d_model, 10)
    def forward(self, x):
        return self.head(self.encoder(x).mean(dim=1))

class BERTBaseEncoder(nn.Module):
    """12-layer Transformer encoder (~85M params, BERT-base scale)."""
    def __init__(self, d_model=768, nhead=12, n_layers=12):
        super().__init__()
        layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=nhead, dim_feedforward=3072,
            batch_first=True, norm_first=True, activation='gelu',
        )
        self.encoder = nn.TransformerEncoder(layer, num_layers=n_layers)
        self.head = nn.Linear(d_model, 10)
    def forward(self, x):
        return self.head(self.encoder(x).mean(dim=1))

# ============================================================
# Benchmark Runner
# ============================================================

def gpu_bench_large(model, inp, label):
    try:
        model.eval()
        with torch.no_grad():
            for _ in range(WARMUP):
                model(inp)
            torch.cuda.synchronize()
            times = []
            for _ in range(ITERS):
                torch.cuda.synchronize()
                t0 = time.perf_counter()
                model(inp)
                torch.cuda.synchronize()
                times.append((time.perf_counter() - t0) * 1000)
        mean_ms = statistics.mean(times)
        std_ms = statistics.stdev(times) if len(times) > 1 else 0
        print(f"    {label}: {mean_ms:.2f} ms (±{std_ms:.2f})")
        return mean_ms
    except Exception as e:
        print(f"    {label}: FAILED — {e}")
        return None

def count_params(model):
    return sum(p.numel() for p in model.parameters())

large_models = [
    ("Deep MLP (6L, 1024d)",     DeepMLP,          (8, 1024)),
    ("GPT-2 Small (4L, 768d)",   GPT2Small,        (8, 128, 768)),
    ("Wide Transformer (6L, 1024d)", WideTransformer, (4, 64, 1024)),
    ("BERT-Base (12L, 768d)",    BERTBaseEncoder,   (4, 128, 768)),
]

large_results = {}
for name, model_cls, shape in large_models:
    m_tmp = model_cls()
    n_params = count_params(m_tmp)
    del m_tmp
    print(f"\n{'='*60}")
    print(f"  {name} — {n_params/1e6:.1f}M params")
    print(f"  Input shape: {shape}")
    print(f"{'='*60}")

    inp = torch.randn(shape, device=device)

    # 1. Vanilla
    torch._dynamo.reset()
    m = model_cls().to(device).eval()
    vanilla = gpu_bench_large(m, inp, "Vanilla GPU")
    del m; torch.cuda.empty_cache(); gc.collect()

    # 2. Inductor
    torch._dynamo.reset()
    m = model_cls().to(device).eval()
    print("    Compiling with Inductor (max-autotune)...")
    try:
        m_ind = torch.compile(m, backend="inductor", mode="max-autotune")
        inductor = gpu_bench_large(m_ind, inp, "Inductor")
    except Exception as e:
        print(f"    Inductor: COMPILE FAILED — {e}")
        inductor = None
    del m; torch.cuda.empty_cache(); gc.collect()

    # 3. Hypatia
    hypatia_ms = None
    if hypatia_core._RUST_AVAILABLE:
        torch._dynamo.reset()
        m = model_cls().to(device).eval()
        print("    Compiling with Hypatia...")
        try:
            m_hyp = torch.compile(m, backend="hypatia")
            hypatia_ms = gpu_bench_large(m_hyp, inp, "Hypatia")
        except Exception as e:
            print(f"    Hypatia: COMPILE FAILED — {e}")
        del m; torch.cuda.empty_cache(); gc.collect()
    else:
        print("    Hypatia: SKIPPED (no Rust core)")

    large_results[name] = {
        "params_m": n_params / 1e6,
        "vanilla": vanilla,
        "inductor": inductor,
        "hypatia": hypatia_ms,
    }

# ============================================================
# Summary Table
# ============================================================
print("\n")
print("=" * 85)
print("  SCALING BENCHMARK: Hypatia vs Inductor — Larger Models (T4 GPU)")
print("=" * 85)
print(f"  {'Model':<32} {'Params':>7} {'Vanilla':>10} {'Inductor':>10} {'Hypatia':>10} {'Hyp/Ind':>8}")
print("  " + "-" * 80)
for name, r in large_results.items():
    params = f"{r['params_m']:.1f}M"
    van = f"{r['vanilla']:.2f}ms" if r['vanilla'] else "N/A"
    ind = f"{r['inductor']:.2f}ms" if r['inductor'] else "N/A"
    hyp = f"{r['hypatia']:.2f}ms" if r['hypatia'] else "N/A"
    if r['inductor'] and r['hypatia']:
        ratio = r['hypatia'] / r['inductor']
        ratio_str = f"{ratio:.2f}x"
        if ratio < 1.0:
            ratio_str += " ✅"
        else:
            ratio_str += " ❌"
    else:
        ratio_str = "N/A"
    print(f"  {name:<32} {params:>7} {van:>10} {ind:>10} {hyp:>10} {ratio_str:>8}")

print()
print("  Hyp/Ind < 1.0 = Hypatia faster ✅ | > 1.0 = Inductor faster ❌")
print("  Value proposition: Hypatia wins on Transformer-heavy architectures")
print("  where E-graph discovers cross-layer GELU+MLP and LN+Attention fusions")

---
## 6. INT4 Block Quantization

Hypatia's Rust-native INT4 quantization with cosine similarity validation.

In [ ]:
from hypatia_core import QuantizedModel
import torch.nn.functional as F

model = nn.Sequential(
    nn.Linear(1024, 2048),
    nn.ReLU(),
    nn.Linear(2048, 1024),
    nn.ReLU(),
    nn.Linear(1024, 10),
).eval()

# QuantizedModel(model, group_size) — INT4 quantization via Rust SIMD
qmodel = QuantizedModel(model, group_size=64)

print("=== INT4 Quantization ===")
fp32_size = sum(p.numel() * 4 for p in model.parameters()) / 1e6
int4_size = fp32_size / qmodel.compression_ratio
print(f"FP32 size: {fp32_size:.1f} MB")
print(f"INT4 size: {int4_size:.1f} MB")
print(f"Compression: {qmodel.compression_ratio:.1f}x")

# Validate
x = torch.randn(1, 1024)
with torch.no_grad():
    y_fp32 = model(x)
    y_int4 = qmodel(x)

cos_sim = F.cosine_similarity(y_fp32.flatten(), y_int4.flatten(), dim=0).item()
max_diff = (y_fp32 - y_int4).abs().max().item()
print(f"\nNumerical Validation:")
print(f"  Cosine similarity: {cos_sim:.6f}")
print(f"  Max absolute diff: {max_diff:.6f}")
print(f"  Status: {'PASS' if cos_sim > 0.99 else 'WARN'} (threshold: 0.99)")

---
## 7. Semantic Validation

Verify that optimized models produce numerically equivalent outputs.

In [ ]:
from hypatia_core import SemanticValidator, optimize

original = nn.Sequential(
    nn.Linear(256, 512),
    nn.ReLU(),
    nn.Linear(512, 256),
    nn.ReLU(),
    nn.Linear(256, 10),
).eval()

optimized = optimize(original)

validator = SemanticValidator(tolerance=1e-5, num_samples=10)
result = validator.validate_models(original, optimized, input_shape=(1, 256))

print("=== Semantic Validation ===")
print(f"Valid: {result['is_valid']}")
print(f"Max absolute diff: {result['max_diff']:.2e}")
print(f"Mean absolute diff: {result['mean_diff']:.2e}")
print(f"Cosine similarity: {result['cosine_similarity']:.8f}")
print(f"\nFloating-point non-associativity means tiny diffs are expected.")
print(f"Tolerance 1e-5 (strict mode) is appropriate for FP32 inference.")

---
## 8. Interactive Benchmark Dashboard

Generate a self-contained HTML report with all benchmark results.

In [ ]:
from hypatia_core import generate_benchmark_dashboard

# Collect results
dashboard_results = {}
for name, r in results.items():
    if r['vanilla']:
        dashboard_results[f"{name} - Vanilla GPU"] = r['vanilla']
    if r['inductor']:
        dashboard_results[f"{name} - Inductor"] = r['inductor']
    if r['hypatia']:
        dashboard_results[f"{name} - Hypatia"] = r['hypatia']

html = generate_benchmark_dashboard(
    model_name="Hypatia Colab Demo",
    results=dashboard_results,
    model_info={
        "n_params": sum(p.numel() for p in TransformerBlock().parameters()),
        "arch": "transformer+mlp",
    },
    tuner_name="Auto (quick_tune)",
    output_path="hypatia_benchmark.html",
)

print(f"Dashboard saved: hypatia_benchmark.html ({len(html):,} chars)")
print("Download from the Files panel on the left.")

In [ ]:
# Display dashboard inline
from IPython.display import HTML, display
escaped = html.replace('&', '&amp;').replace('"', '&quot;')
display(HTML(f'<iframe srcdoc="{escaped}" width="100%" height="600" style="border:1px solid #333; border-radius:8px;"></iframe>'))

---
## 9. torch.compile Integration (Drop-in Backend)

Hypatia works as a drop-in `torch.compile` backend. **Zero code changes** to your model.

In [ ]:
if hypatia_core._RUST_AVAILABLE:
    model = nn.Sequential(
        nn.Linear(512, 1024),
        nn.GELU(),
        nn.Linear(1024, 512),
        nn.LayerNorm(512),
        nn.Linear(512, 10),
    ).to(device).eval()

    # One line to optimize:
    optimized = torch.compile(model, backend='hypatia')

    x = torch.randn(4, 64, 512, device=device)
    with torch.no_grad():
        y = optimized(x)

    print(f"Input shape:  {x.shape}")
    print(f"Output shape: {y.shape}")
    print(f"\nHypatia pipeline:")
    print(f"  Phase 1: E-graph structural optimization (Rust)")
    print(f"  Phase 2: Triton kernel fusion (torch.compile/inductor)")
    print(f"\nDone! Model optimized with zero code changes.")
else:
    print("Skipping: Rust core required for torch.compile backend.")
    print("The Python-only features (profiler, auto-tuner, dashboard, fused modules)")
    print("all work without the Rust core.")

---
## Summary

| Feature | Status | Requires Rust |
|---------|--------|---------------|
| E-Graph equality saturation | 37 rewrite rules, <1ms saturation | Yes |
| torch.compile backend | Drop-in, zero code changes | Yes |
| GPU fused kernels | Linear+ReLU, GELU+MLP, Attention, LayerNorm | No |
| INT4 quantization | 5.3-6.4x compression, >99.5% cosine similarity | No |
| Auto-tuner | Hardware-aware strategy selection (<200ms) | No |
| FLOPs profiler | Shape-aware with roofline analysis | No |
| Semantic validation | Configurable strict/soft/off | No |
| HTML dashboard | Self-contained, dark theme, animated | No |

### Key Finding: Hypatia vs Inductor

- **Transformer blocks**: Hypatia ~14% faster (E-graph discovers cross-layer GELU+MLP fusion)
- **Standard MLPs**: Inductor faster (greedy patterns already optimal)
- **Value proposition**: Cross-layer fusion discovery for complex architectures

### Links

- [GitHub Repository](https://github.com/mahreman/hypatia)
- [Academic Paper](https://github.com/mahreman/hypatia/blob/main/docs/hypatia_paper.md)
- [User Guide](https://github.com/mahreman/hypatia/blob/main/hypatia_core/docs/USER_GUIDE.md)

**Citation:**
```bibtex
@software{hypatia2025,
  author = {Eryol, Ender},
  title = {Hypatia: Hardware-Aware Symbolic Compilation via E-Graph Equality Saturation},
  year = {2025},
  url = {https://github.com/mahreman/hypatia}
}
```